# state_conditioned_drawdown_rebound_reward_train_v1

Runtime card: `runtime/reward/card/state_conditioned_drawdown_rebound_reward_train_v1`

Purpose: run one governed training branch that moves the failed drawdown/rebound trade-off upstream into the reward function. The branch adds state-conditioned shaping to the shared reward path: broken-anchor/high-vol downtrend states penalize excess exposure, while high-vol positive-momentum rebound states penalize underparticipation.

Governance:
- **Direction**: `reward`, with model-selection and walk-forward evidence.
- **Canonical baseline**: `risk_budget_usage_v1` family and current runtime composer evidence.
- **Direct comparison baseline**: `mode_conditioned_rebound_reward_train_v2` because it already tried mode-conditioned reward shaping but remained underparticipated and high-drawdown.
- **Role**: `ablate` / `diagnose`; no runtime promotion unless strict and walk-forward gates pass.
- **Changed layer**: `reward`; shared capability lives in `src/soxl_quant/policy/reward.py` and `src/soxl_quant/config.py`; this notebook owns the special profile and conclusion.
- **Target**: fixed proxy relative return `>= +30pp`, fixed proxy MaxDD `<= 15%`, every saved walk-forward window relative return `>= +30pp`, and every saved walk-forward MaxDD `<= 15%`.
- **Expected trade-off**: better 2022/downtrend exposure without repeating the global low-position collapse; rebound opportunity should reduce 2025/2026 underparticipation.
- **Failure mode**: strict MaxDD improves but `wf_2020`/rebound participation collapses, or rebound improves while 2022/downtrend drawdown expands.
- **Comparison-table row key**: `state_conditioned_drawdown_rebound_reward_train_v1`.

Historical positioning:
- **Closest success**: `drawdown_aware_objective_screen_v1` found strict low-drawdown candidates with `>=+30pp` relative return and `<15%` MaxDD, proving the target is reachable on some fixed paths.
- **Closest failure**: `mode_conditioned_rebound_reward_train_v2` added mode-conditioned reward shaping but still failed strict MaxDD and underparticipated in proxy/recovery windows.
- **Why this is not a duplicate**: this branch adds explicit feature-state conditions from the poor-window evidence (`vol_of_vol_21`, `tracking_error_vol_21`, `decay_63`, momentum/rebound states) rather than another scalar/global drawdown penalty or another selector over existing actors.


## External Basis

Primary sources used to constrain this run:
- Qlib: keep the experiment inside the reproducible qlib/RL harness rather than adding a parallel workflow. https://github.com/microsoft/qlib
- FinRL: use train/test style portfolio RL discipline and explicit evaluation artifacts. https://github.com/AI4Finance-Foundation/FinRL
- DeepTrader: market-condition embeddings and maximum-drawdown-aware reward design support state-conditioned risk terms. https://arxiv.org/abs/2109.08712
- Bailey / Borwein / Lopez de Prado / Zhu PBO and Deflated Sharpe work: strict fixed-window success is not enough; walk-forward and overfit checks must decide promotion. https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf


In [1]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

import pandas as pd

PROFILE = 'state_conditioned_drawdown_rebound_reward_train_v1'
ROOT = Path.cwd()
CARD = ROOT / 'runtime' / 'reward' / 'card' / PROFILE
MODEL_CARD = ROOT / 'runtime' / 'model' / 'card' / PROFILE
SUMMARY_PATH = MODEL_CARD / f'soxl_{PROFILE}_summary.json'
SELECTION_PATH = CARD / 'selection.json'
LOG_PATH = CARD / 'train_walkforward.log'
TARGET_RELATIVE_PP = 30.0
TARGET_MAX_DD = 15.0
FORCE_TRAIN = False

CARD.mkdir(parents=True, exist_ok=True)
MODEL_CARD.mkdir(parents=True, exist_ok=True)
print({'card': str(CARD.relative_to(ROOT)), 'summary_exists': SUMMARY_PATH.exists()})


{'card': 'runtime/reward/card/state_conditioned_drawdown_rebound_reward_train_v1', 'summary_exists': False}


In [2]:
cmd = [
    str(ROOT / 'qlib_venv' / 'bin' / 'python'),
    '-m',
    'src.soxl_quant.app',
    '--profiles',
    PROFILE,
    '--skip-data-refresh',
    '--walk-forward',
]

if FORCE_TRAIN or not SUMMARY_PATH.exists():
    env = dict(os.environ)
    env['PYTHONPATH'] = str(ROOT)
    result = subprocess.run(cmd, cwd=ROOT, env=env, text=True, capture_output=True, timeout=7200)
    LOG_PATH.write_text(result.stdout + '\n\n[stderr]\n' + result.stderr, encoding='utf-8')
    print('\n'.join((result.stdout + result.stderr).splitlines()[-80:]))
    if result.returncode != 0:
        raise RuntimeError(f'training command failed with exit code {result.returncode}; see {LOG_PATH.relative_to(ROOT)}')
else:
    print(f'skipped training because {SUMMARY_PATH.relative_to(ROOT)} already exists')


[34681:MainThread](2026-05-25 14:40:37,465) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] rew = -329.66019705483353
[34681:MainThread](2026-05-25 14:40:37,465) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] len = 299.0
[34681:MainThread](2026-05-25 14:40:37,465) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] rew_std = 99.10505200191903
[34681:MainThread](2026-05-25 14:40:37,466) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] len_std = 0.0
[34681:MainThread](2026-05-25 14:40:37,466) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] loss = 0.06088643468211632
[34681:MainThread](2026-05-25 14:40:37,466) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] loss/clip = -0.002408705928639786
[34681:MainThread](2026-05-25 14:40:37,466) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] loss/vf = 0.3173479911711839
[34681:MainThread](2026-05-25 14:40:37,466) INFO - qlib.rl.trainer.vessel - [vessel.py:83] - [Iter 3] loss/ent = 1.6

In [3]:
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
previous_paths = {
    'drawdown_rebound_reward_train_v1': ROOT / 'runtime/reward/card/drawdown_rebound_reward_train_v1/selection.json',
    'mode_conditioned_rebound_reward_train_v2': ROOT / 'runtime/reward/card/mode_conditioned_rebound_reward_train_v2/selection.json',
}
previous = {name: json.loads(path.read_text(encoding='utf-8')) for name, path in previous_paths.items() if path.exists()}

summary.keys(), list(previous)


(dict_keys(['profile', 'seed', 'feature_schema', 'position_cap', 'position_budget', 'selection_score', 'checkpoint_retention', 'model_config', 'sampling_config', 'warm_start', 'model_path', 'retained_checkpoints', 'valid', 'best_score', 'selection_history', 'slices', 'test_proxy', 'promotion']),
 ['drawdown_rebound_reward_train_v1',
  'mode_conditioned_rebound_reward_train_v2'])

In [4]:
def metric_row(label: str, metrics: dict) -> dict:
    return {
        'label': label,
        'return': float(metrics.get('return', 0.0)),
        'buy_and_hold_return': float(metrics.get('buy_and_hold_return', 0.0)),
        'relative_return': float(metrics.get('relative_return', metrics.get('return', 0.0) - metrics.get('buy_and_hold_return', 0.0))),
        'sharpe': float(metrics.get('sharpe', 0.0)),
        'max_dd': float(metrics.get('max_dd', 0.0)),
        'avg_pos': float(metrics.get('avg_pos', 0.0)),
        'stress_avg_pos': float(metrics.get('stress_avg_pos', 0.0)),
        'trades': float(metrics.get('trades', 0.0)),
    }

rows = [metric_row('test_proxy', summary['test_proxy'])]
for slice_name in ['test_2020', 'test_2022', 'test_2023_2024']:
    rows.append(metric_row(slice_name, summary.get('slices', {}).get(slice_name, {})))

wf_rows = []
for wf_name in ['wf_2020', 'wf_2022', 'wf_2023_2024']:
    wf_summary = MODEL_CARD / f'soxl_{PROFILE}_{wf_name}_summary.json'
    if wf_summary.exists():
        payload = json.loads(wf_summary.read_text(encoding='utf-8'))
        wf_rows.append(metric_row(wf_name, payload.get('test_proxy', {})))
    else:
        wf_rows.append({'label': wf_name, 'missing': True})

metrics = pd.DataFrame(rows)
walkforward = pd.DataFrame(wf_rows)
metrics.to_csv(CARD / 'metrics_summary.csv', index=False)
walkforward.to_csv(CARD / 'walkforward_summary.csv', index=False)
metrics, walkforward


(            label      return  buy_and_hold_return  relative_return    sharpe  \
 0      test_proxy  619.708826          2542.027442     -1470.032017  1.016311   
 1       test_2020   80.646866           355.736160      -275.089294  1.965683   
 2       test_2022  -36.906505           -76.614257        52.906964 -1.229305   
 3  test_2023_2024   56.176271            55.654474         0.521797  0.925303   
 
       max_dd   avg_pos  stress_avg_pos  trades  
 0  43.565118  0.382524        0.203955     0.0  
 1  23.985607  0.431536        0.194797     0.0  
 2  41.483544  0.258405        0.179244     0.0  
 3  27.250178  0.370570        0.227380     0.0  ,
           label     return  buy_and_hold_return  relative_return    sharpe  \
 0       wf_2020  56.775553           355.736160      -275.089294  1.412174   
 1       wf_2022 -23.275008           -76.614257        45.125927 -0.727134   
 2  wf_2023_2024  -3.160370           -39.952710        36.987080 -0.171970   
 
       max_dd   avg

In [5]:
def pass_metric(row: dict, *, rel_floor: float = TARGET_RELATIVE_PP, dd_cap: float = TARGET_MAX_DD) -> bool:
    return float(row.get('relative_return', -1e9)) >= rel_floor and float(row.get('max_dd', 1e9)) <= dd_cap

main = metric_row('test_proxy', summary['test_proxy'])
wf_records = walkforward.to_dict(orient='records')
strict_pass = pass_metric(main)
wf_passes = {r['label']: (not r.get('missing', False)) and pass_metric(r) for r in wf_records}
full_pass = strict_pass and all(wf_passes.values())

blockers = []
if main['relative_return'] < TARGET_RELATIVE_PP:
    blockers.append(f"test_proxy relative {main['relative_return']:.2f}pp < {TARGET_RELATIVE_PP:.2f}pp")
if main['max_dd'] > TARGET_MAX_DD:
    blockers.append(f"test_proxy MaxDD {main['max_dd']:.2f}% > {TARGET_MAX_DD:.2f}%")
for row in wf_records:
    label = row['label']
    if row.get('missing', False):
        blockers.append(f'{label} summary missing')
        continue
    if row['relative_return'] < TARGET_RELATIVE_PP:
        blockers.append(f"{label} relative {row['relative_return']:.2f}pp < {TARGET_RELATIVE_PP:.2f}pp")
    if row['max_dd'] > TARGET_MAX_DD:
        blockers.append(f"{label} MaxDD {row['max_dd']:.2f}% > {TARGET_MAX_DD:.2f}%")

decision = 'promote_candidate' if full_pass else 'diagnostic_no_promotion'
selection = {
    'profile': PROFILE,
    'decision': decision,
    'target': {'relative_return_min_pp': TARGET_RELATIVE_PP, 'max_dd_max_pct': TARGET_MAX_DD},
    'strict_pass': strict_pass,
    'walkforward_passes': wf_passes,
    'full_pass': full_pass,
    'blockers': blockers,
    'metrics': {'test_proxy': main, 'walkforward': wf_records},
    'promotion': summary.get('promotion', {}),
    'comparison_baselines': {
        'drawdown_rebound_reward_train_v1': previous.get('drawdown_rebound_reward_train_v1', {}).get('decision'),
        'mode_conditioned_rebound_reward_train_v2': previous.get('mode_conditioned_rebound_reward_train_v2', {}).get('decision'),
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2109.08712',
        'https://www.davidhbailey.com/dhbpapers/deflated-sharpe.pdf',
    ],
    'artifacts': {
        'notebook': str((CARD / f'{PROFILE}.ipynb').relative_to(ROOT)),
        'profile': str((CARD / 'profile.yaml').relative_to(ROOT)),
        'train_log': str(LOG_PATH.relative_to(ROOT)),
        'summary': str(SUMMARY_PATH.relative_to(ROOT)),
        'metrics_summary': str((CARD / 'metrics_summary.csv').relative_to(ROOT)),
        'walkforward_summary': str((CARD / 'walkforward_summary.csv').relative_to(ROOT)),
        'selection': str(SELECTION_PATH.relative_to(ROOT)),
    },
}

SELECTION_PATH.write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding='utf-8')
selection


{'profile': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'decision': 'diagnostic_no_promotion',
 'target': {'relative_return_min_pp': 30.0, 'max_dd_max_pct': 15.0},
 'strict_pass': False,
 'walkforward_passes': {'wf_2020': False,
  'wf_2022': False,
  'wf_2023_2024': False},
 'full_pass': False,
 'blockers': ['test_proxy relative -1470.03pp < 30.00pp',
  'test_proxy MaxDD 43.57% > 15.00%',
  'wf_2020 relative -275.09pp < 30.00pp',
  'wf_2020 MaxDD 24.41% > 15.00%',
  'wf_2022 MaxDD 29.15% > 15.00%',
  'wf_2023_2024 MaxDD 15.23% > 15.00%'],
 'metrics': {'test_proxy': {'label': 'test_proxy',
   'return': 619.7088261372328,
   'buy_and_hold_return': 2542.0274416989405,
   'relative_return': -1470.032016846747,
   'sharpe': 1.0163109758465843,
   'max_dd': 43.56511845576983,
   'avg_pos': 0.38252412562732946,
   'stress_avg_pos': 0.20395510754249174,
   'trades': 0.0},
  'walkforward': [{'label': 'wf_2020',
    'return': 56.77555264698506,
    'buy_and_hold_return': 355.736159725

## Runtime Conclusion

The cell above is the authoritative single-run conclusion for this card. It records whether the state-conditioned reward branch passed the hard `+30pp / <=15%` fixed-proxy and walk-forward gates, plus the concrete blockers if it did not. Do not promote this profile or update durable routing unless `full_pass` is true and the promotion sidecar agrees.
